# SAAM Project - Portfolio Allocation with a Carbon Objective

**Region:** Pacific. **Carbon scope:** Scope 1. **Implementation window:** 2014-01 to 2025-12.

This notebook reproduces every table and figure used in our report. Run all cells from top to bottom. All paths are relative to the repo root - nothing is hard-coded to a local machine.

Heavy lifting (cleaning, MV, VW, carbon constraints, NZ trajectory) lives in `src/01_cleaning.py`, `src/02_analysis.py`, `src/05_part3.py`, `src/06_part4.py` and the shared helpers in `src/saam_core.py`. This notebook just orchestrates them and renders the results.

---

**Use of Large Language Models (LLMs).** We used ChatGPT / Claude to help debug a few pandas slicing issues and to suggest matplotlib styling for the carbon-trajectory plot. All methodological choices, the optimisation problems, the construction of the eligible universe, the implementation of the carbon constraints, and the interpretation of the results are our own. We are fully responsible for the correctness of the output.

## 0. Setup

In [ ]:
import sys, runpy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

# Find the repo root from the notebook location (works whether the notebook is
# opened from src/ or from the project root).
_here = Path.cwd()
ROOT = _here if (_here / 'src').exists() else _here.parent
SRC = ROOT / 'src'
DATA = ROOT / 'data' / 'processed'
OUTPUTS = ROOT / 'outputs'
TABLES = OUTPUTS / 'tables'
FIGURES = OUTPUTS / 'figures'

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print('ROOT =', ROOT)

## Part I - Standard portfolio allocation

Data cleaning (PDF Section 2.1) and the long-only minimum-variance optimisation (Section 2.2) are implemented in `src/01_cleaning.py`, `src/02_analysis.py` and `src/MVP-construction.ipynb`. Their monthly outputs sit in `data/processed/`. We just load and summarise them here.

Key choices used in Part I:
- estimation window of 120 months (10 years) ending in December of each allocation year;
- exclude firms with fewer than 3 years of returns, or more than 50% stale (zero-return) months;
- exclude firms with no carbon data at end-Y;
- treat prices below 0.5 as missing, forward-fill mid-sample gaps, set the price to 0 at delisting (-100% return that month);
- SLSQP, long-only, weights sum to 1, rebalanced once a year.

In [ ]:
mv_raw = pd.read_csv(DATA / 'mv_portfolio_returns.csv', parse_dates=['Date']).set_index('Date')
mv_returns = mv_raw['Return']
mv_returns.head()

In [ ]:
# Pacific monthly risk-free rate (RF in percent/month -> fraction). Used for the Sharpe.
rf_raw = pd.read_csv(DATA / 'rf_rate.csv')
rf_raw.columns = ['period', 'rf']
rf_idx = pd.to_datetime(rf_raw['period'].astype(int).astype(str), format='%Y%m') + pd.offsets.MonthEnd(0)
rf_monthly = pd.Series((rf_raw['rf'] / 100).to_numpy(), index=rf_idx).sort_index()
rf_monthly.tail()

In [ ]:
def summary(monthly: pd.Series, rf: pd.Series | None = None) -> pd.Series:
    r = monthly.dropna()
    ann_ret = (1 + r).prod() ** (12 / len(r)) - 1
    ann_vol = r.std(ddof=0) * np.sqrt(12)
    ann_rf = rf.reindex(r.index).dropna().mean() * 12 if rf is not None else 0.0
    return pd.Series({
        'ann_return':        ann_ret,
        'ann_volatility':    ann_vol,
        'ann_risk_free':     ann_rf,
        'sharpe (excess)':   (ann_ret - ann_rf) / ann_vol,
        'min monthly':       r.min(),
        'max monthly':       r.max(),
        'cumulative return': (1 + r).prod() - 1,
    })

mv_stats = summary(mv_returns, rf_monthly)
mv_stats.to_frame('P_mv_oos')

## Part II - Value-weighted benchmark

PDF Section 2.3: $R^{(vw)}_{t+1} = \sum_i w_{i,t} R_{i,t+1}$ with $w_{i,t} = \mathrm{Cap}_{i,t} / \sum_j \mathrm{Cap}_{j,t}$. We compute it in `src/vwp.ipynb`; the monthly series is cached on disk.

In [ ]:
vw_raw = pd.read_csv(DATA / 'vw_portfolio_returns.csv', index_col=0, parse_dates=True)
vw_returns = vw_raw.iloc[:, 0]
vw_returns.name = 'VW'
vw_returns.head()

In [ ]:
vw_stats = summary(vw_returns, rf_monthly)
pd.concat({'P_mv_oos': mv_stats, 'P_vw': vw_stats}, axis=1)

In [ ]:
# Cumulative growth of $1 - Parts I vs II.
fig, ax = plt.subplots(figsize=(10, 5))
(1 + mv_returns).cumprod().plot(ax=ax, label='P_mv_oos', linewidth=2)
(1 + vw_returns).cumprod().plot(ax=ax, label='P_vw',     linewidth=2)
ax.set_title('Parts I & II - cumulative growth of $1 (2014-2025)')
ax.set_ylabel('Growth of $1')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

Comment. The unconstrained long-only MV portfolio earns substantially more than the cap-weighted benchmark over 2014-2025 (cumulative +60% vs +18%) at lower volatility - consistent with the well-known low-vol anomaly on developed equities. The benchmark drawdown profile is also wider (deeper monthly minima).

## Part III - 50% carbon-footprint reduction

Runs `src/05_part3.py` (which uses `src/saam_core.py`). The script:
- rebuilds the eligible universe each year (carbon + revenue + cap available, price > 0, $\ge$ 36 obs in the 120-month window, $\le$ 50% stale months);
- solves three long-only QPs per year via SLSQP with analytic gradients:
  - $P^{(mv)}_{oos}$: min $w^\top \Sigma w$ s.t. $w \ge 0$, $\sum w = 1$
  - $P^{(mv)}_{oos}(0.5)$: same + $CF \le 0.5 \cdot CF(P^{(mv)}_{oos})$
  - $P^{(vw)}_{oos}(0.5)$: min $(w-w_{vw})^\top \Sigma (w-w_{vw})$ + $CF \le 0.5 \cdot CF(P^{(vw)})$
- simulates each implementation year with the drift formula from PDF Section 2.2.

Outputs are written to `outputs/tables/` and `outputs/figures/`.

In [ ]:
# This call prints the per-year CF for each strategy. Takes ~30s.
runpy.run_path(str(SRC / '05_part3.py'), run_name='__main__')
print('Part III done.')

### 3.1 Carbon metrics - $P^{(mv)}_{oos}$ vs $P^{(vw)}$

In [ ]:
pd.read_csv(TABLES / 'part3_carbon_metrics_mv_vw.csv')

In [ ]:
display(Image(filename=str(FIGURES / 'part3_waci_mv_vs_vw.png')))
display(Image(filename=str(FIGURES / 'part3_carbon_footprint_mv_vs_vw.png')))

Top-10 firms driving the carbon footprint and WACI of the VW benchmark each year (uses $w^{vw}_i \cdot \mathrm{E}_i / \mathrm{Cap}_i$ and $w^{vw}_i \cdot \mathrm{CI}_i$ respectively). Mining + utilities dominate the early years; integrated steel / energy take over later.

In [ ]:
pd.read_csv(TABLES / 'part3_top10_cf_contributors.csv').head(15)

In [ ]:
pd.read_csv(TABLES / 'part3_top10_waci_contributors.csv').head(15)

### 3.2 $P^{(mv)}_{oos}(0.5)$ - min variance with CF $\le 0.5 \cdot CF(P^{(mv)}_{oos})$

In [ ]:
pd.read_csv(TABLES / 'part3_summary_mv_vs_mv_carbon50.csv')

In [ ]:
# Slack = cap - realised CF. Should be ~0 (binding) every year.
pd.read_csv(TABLES / 'part3_constraint_slack_mv_carbon50.csv')[
    ['year', 'carbon_footprint', 'carbon_limit', 'slack', 'optimization_success']
]

In [ ]:
for fn in ['part3_cumulative_mv_vs_mv_carbon50.png',
           'part3_cf_mv_vs_mv_carbon50.png',
           'part3_waci_mv_vs_mv_carbon50.png']:
    display(Image(filename=str(FIGURES / fn)))

Comment. Imposing a 50% cap on the MV carbon footprint does NOT cost return on this sample - on the contrary, $P^{(mv)}_{oos}(0.5)$ delivers a higher annualised return (9.06% vs 8.30%) and a higher Sharpe (0.62 vs 0.57). Volatility is nearly identical. The optimiser reallocates weight from the carbon-intense low-vol firms (utilities, steel) towards similarly low-vol but lower-CF names (industrials, IT, healthcare). The cap is binding every year.

### 3.3 $P^{(vw)}_{oos}(0.5)$ - tracking-error min with CF $\le 0.5 \cdot CF(P^{(vw)})$

In [ ]:
pd.read_csv(TABLES / 'part3_summary_vw_vs_vw_carbon50.csv')

In [ ]:
pd.read_csv(TABLES / 'part3_tracking_error_vw_carbon50.csv')

In [ ]:
pd.read_csv(TABLES / 'part3_constraint_slack_vw_carbon50.csv')[
    ['year', 'carbon_footprint', 'carbon_limit', 'slack', 'optimization_success']
]

In [ ]:
for fn in ['part3_cumulative_vw_vs_vw_carbon50.png',
           'part3_cf_vw_vs_vw_carbon50.png',
           'part3_waci_vw_vs_vw_carbon50.png',
           'part3_tracking_error_vw_carbon50.png']:
    display(Image(filename=str(FIGURES / fn)))

### 3.4 Trade-off

Two observations across both decarbonised strategies on this sample:
1. Halving the carbon footprint of either reference portfolio does not reduce returns. Both 50%-cap variants finish above their reference portfolio on cumulative return and Sharpe.
2. The 50%-cap constraint binds every single year for both MV(0.5) and VW(0.5). The optimiser sits exactly on the boundary - meaning the carbon objective is the active constraint, not numerical slack.

The tracking error of VW(0.5) vs VW is around 3.2% annualised - a budget any passive investor with a soft mandate would tolerate. MV(0.5) has TE ~9% vs VW, but that is structural (MV is far from cap-weights by construction) and is not the optimisation target.

Detailed weight tables are in `outputs/tables/part3_weights_{mv,vw}_carbon50.csv`. Headline composition shifts: AU mining majors (BHP, Rio Tinto) and Japanese integrated steel are systematically underweighted; healthcare, IT, services pick up weight.

## Part IV - Net-Zero trajectory

Runs `src/06_part4.py`. Same TE optimisation as 3.3 but with a *dynamic* carbon budget:
$$C_Y = (1-\theta)^{Y - Y_0 + 1} \cdot CF(P^{(vw)})_{Y_0}, \quad Y_0 = 2013, \quad \theta = 10\%.$$
The 2013 anchor is computed once and never recomputed. Cap tightens 10% per year through 2024.

Note: this script reads the VW(0.5) cache produced by Part 3 above, so the joint VW / VW(0.5) / VW(NZ) panel agrees with what Part 3 already wrote. If you skip Part 3 it will refuse to run - clear error.

In [ ]:
runpy.run_path(str(SRC / '06_part4.py'), run_name='__main__')
print('Part IV done.')

### Summary - VW vs VW(0.5) vs VW(NZ)

In [ ]:
pd.read_csv(TABLES / 'part4_summary_vw_vs_carbon50_vs_netzero.csv')

### 4.1 Target NZ cap path vs realised CF

In [ ]:
path = pd.read_csv(TABLES / 'part4_netzero_target_vs_realized_cf.csv')
path[['year', 'anchor_cf_vw_2013', 'carbon_limit_nz',
      'carbon_footprint_vw', 'carbon_footprint_vw_nz',
      'slack', 'feasible_within_1e-6']]

In [ ]:
display(Image(filename=str(FIGURES / 'part4_cf_target_vs_realized.png')))

Observation. The cap binds from 2017 onwards. In 2013-2016 the realised CF of VW(NZ) is already below the cap - the TE-optimal portfolio just *happens* to be low-carbon because there are still plenty of low-CF firms with weight close to their VW shares. From 2017 the cap drives the allocation: VW(NZ) sits exactly on the cap each year.

### 4.2 Cumulative performance, WACI, tracking error

In [ ]:
for fn in ['part4_cumulative_vw_vs_carbon50_vs_netzero.png',
           'part4_waci_vw_vs_carbon50_vs_netzero.png',
           'part4_tracking_error_vw_netzero.png']:
    display(Image(filename=str(FIGURES / fn)))

In [ ]:
# TE table per year.
pd.read_csv(TABLES / 'part4_tracking_error_vw_netzero.csv')

In [ ]:
pd.read_csv(TABLES / 'part4_constraint_slack_vw_netzero.csv')[
    ['year', 'carbon_footprint', 'carbon_limit', 'slack', 'optimization_success']
]

Cost of net-zero. Over 2014-2025, VW(NZ) earns 7.70% annualised vs 7.79% for VW(0.5) and 6.72% for VW. So the dynamic cap costs ~9 bps/yr versus the static 50% cut but still beats VW by ~1 pp/yr. Sharpe drops only marginally (0.437 vs 0.444). Ex-post TE is 3.16% - actually slightly *below* VW(0.5)'s 3.24%, although ex-ante TE rises mechanically as the cap tightens (see the TE plot - from ~2.5% in 2013 to ~4.5% in 2024).

The strategy is feasible every year on this universe (min slack at the level of numerical noise). It would become harder if (i) the anchor were tighter, (ii) the eligible universe shrank, or (iii) $\theta$ were larger.

## 5. Validation

Quick sanity checks. Detailed checklist with file-level provenance is in `outputs/validation_checklist_part3_part4.md`.

In [ ]:
# Weights sum to 1, no negatives, no NaNs - for both Part 3 and Part 4.
for src_file in ['part3_weights_mv_carbon50.csv',
                 'part3_weights_vw_carbon50.csv',
                 'part4_weights_vw_netzero.csv']:
    w = pd.read_csv(TABLES / src_file)
    agg = w.groupby('year')['weight'].agg(['sum', 'min', lambda s: s.isna().sum()])
    agg.columns = ['sum', 'min', 'n_nan']
    print(f'\n{src_file}')
    print(agg.round(8).to_string())

In [ ]:
# All carbon constraints satisfied within 1e-6?
for src_file in ['part3_constraint_slack_mv_carbon50.csv',
                 'part3_constraint_slack_vw_carbon50.csv',
                 'part4_constraint_slack_vw_netzero.csv']:
    s = pd.read_csv(TABLES / src_file)
    worst = s['slack'].min()
    ok    = s['optimization_success'].all()
    print(f'{src_file:50s}  min slack = {worst:+.3e}   all solves ok? {ok}')

## 6. Notes & limitations

Five issues that we flag in the report (Section 5.2 / criterion v):

1. **Data coverage before 2010 is thin.** That is exactly why the PDF starts the allocation in 2013. We keep the 120-month estimation window even for the 2013 decision, which means many firms with patchy 2004-2008 histories drop out of the eligible set. Sensitivity to this filter is meaningful.
2. **Covariance is sample-based and not shrunk.** With $\sim$ 300-470 firms and 120 monthly observations, the sample covariance is rank-deficient or near so. We add a tiny ridge $\epsilon = 10^{-6} I$ for numerical safety. Production work would use Ledoit-Wolf or factor shrinkage; results would shift but the *direction* of the carbon trade-off would not.
3. **Forward-fill of missing carbon / revenues**, per the PDF. This biases recent observations of a firm that stopped reporting - which is exactly when its true CF may have changed.
4. **Constraint feasibility in early years.** The NZ cap was non-binding in 2013-2015 because the cap path starts loose. If the anchor year were 2010 or if $\theta$ were 15-20%, the cap could become infeasible in late years - the optimiser would force the constraint to the boundary at the expense of TE.
5. **Look-ahead.** We use only end-of-Y carbon/revenues/cap to decide the Y+1 allocation, but in practice carbon reports for year Y are often disclosed in mid-Y+1. A more conservative implementation would lag emissions by one extra year - we have not done that here.

---

End of notebook. Tables: `outputs/tables/`. Figures: `outputs/figures/`. Interpretation notes: `outputs/interpretation_part3.md`, `outputs/interpretation_part4.md`.